# Story 1.1: Agent Integration Test

Simple test to verify:
1. Scenario Generator Agent creates scenarios
2. Simulation Feedback Agent provides feedback
3. MLflow traces are logged to summit-sim experiment

## 1. Setup & Imports

In [1]:
import mlflow

from summit_sim.agents.generator import generate_scenario
from summit_sim.agents.simulation import process_choice
from summit_sim.schemas import HostConfig
from summit_sim.settings import settings

# Initialize MLflow
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
mlflow.pydantic_ai.autolog()

print(f"✓ MLflow tracking URI: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment name: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key configured: {bool(settings.openrouter_api_key)}")

✓ MLflow tracking URI: https://mlflow.bhamm-lab.com
✓ Experiment name: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key configured: True


## 2. Test Scenario Generator

In [2]:
# Test configuration
config = HostConfig(num_participants=3, activity_type="hiking", difficulty="med")

print(f"Generating scenario: {config}")
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Starting turn: {scenario.starting_turn_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating scenario: num_participants=3 activity_type='hiking' difficulty='med'
------------------------------------------------------------
✓ Title: Alpine Fatigue or Something More?
✓ Setting: Alpine trail, 9,000 ft elevation, 4 miles from the trailhead. Sunny, 75°F. The group has been hiking for 5 hours.
✓ Patient: 34-year-old male, experienced hiker. Presents with headache, mild nausea, fatigue, and slight neurological confusion (slow speech). No allergies or chronic medical conditions.
✓ Turns: 3
✓ Starting turn: assessment

Learning Objectives:
  • Perform a comprehensive patient assessment in a wilderness setting.
  • Recognize the signs and symptoms of Exercise-Associated Hyponatremia (EAH) vs. Hypoglycemia.
  • Develop appropriate intervention strategies for EAH while evaluating evacuation necessity.


Trace(trace_id=tr-796f71c04cdfab5df058c2ce57d745ee)

## 3. Test Simulation Feedback (Single Turn)

In [3]:
# Get starting turn
current_turn = scenario.get_turn(scenario.starting_turn_id)

print(f"Turn {current_turn.turn_id}:")
print(f"{current_turn.narrative_text}")
print("\nChoices:")
for i, choice in enumerate(current_turn.choices):
    print(f"  {i + 1}. {choice.description}")

# Select first choice
selected_choice = current_turn.choices[0]
print(f"\n>>> Selecting: {selected_choice.description}")
print("-" * 60)

# Get feedback
result = await process_choice(scenario, current_turn, selected_choice)

print("\nFeedback:")
print(result.feedback)
print("\nLearning Moments:")
for moment in result.learning_moments:
    print(f"  • {moment}")
print(f"\n✓ Scenario complete: {result.is_complete}")

Turn assessment:
You are hiking with a group of three. Your friend, Mark, stops suddenly, complaining of a splitting headache and nausea. When you check on him, he is sitting on a rock, looking pale and acting slightly disoriented. He has been complaining about how much water he's drank all day to "stay ahead of the game."

Choices:
  1. Perform a thorough physical assessment, including vital signs (mental status, breathing), and gather a detailed hydration/nutrition history.
  2. Assume he is dehydrated from the heat, force him to drink 1 liter of fresh water immediately, and force-feed him a sugary energy bar.

>>> Selecting: Perform a thorough physical assessment, including vital signs (mental status, breathing), and gather a detailed hydration/nutrition history.
------------------------------------------------------------

Feedback:
Excellent choice. Jumping to conclusions without a systematic assessment is a common trap in wilderness medicine. By pausing to conduct a proper SOAP e

Trace(trace_id=tr-bd22ec9d09e20e47990d54fcae95877a)

## 4. Test Multiple Activities (Optional)

In [4]:
# Quick test of all activities
activities = ["canyoneering", "skiing", "hiking"]

for activity in activities:
    config = HostConfig(num_participants=2, activity_type=activity, difficulty="low")
    test_scenario = await generate_scenario(config)
    print(f"✓ {activity}: {test_scenario.title[:50]}...")

✓ canyoneering: The Beginner's Slip at Slot Creek...
✓ skiing: The Late-Day Alpine Ankle Sprain...
✓ hiking: Root of the Problem...


[Trace(trace_id=tr-8e62c6e83955ec848828a6c9990274d1), Trace(trace_id=tr-2364ae2c9fc293d17c3e2835f931f9af), Trace(trace_id=tr-0a765eea2b260ea0e3bbb6c40a23217c)]

## 5. MLflow Verification

Check your MLflow UI at the tracking URI to verify:
- Traces appear under the `summit-sim` experiment
- Each agent call is logged with latency metrics
- Token usage is tracked

In [5]:
# Show experiment info
experiment = mlflow.get_experiment_by_name(settings.mlflow_experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"\nView traces at: {settings.mlflow_tracking_uri}")

Experiment ID: 7
Artifact Location: mlflow-artifacts:/7

View traces at: https://mlflow.bhamm-lab.com


---
## ✅ Integration Test Complete

If you see checkmarks and no errors, Story 1.1 is working!